In [1]:
import os
from pathlib import Path
import json
import pandas as pd
from mri_data import utils
import nibabel as nib
from statistics import mean
import pyperclip
import sys
sys.path.append("/home/srs-9/Projects/prl_project")
from helpers.shell_interface import command, run_if_missing, open_itksnap_workspace_cmd


In [16]:
dataroot = Path("/media/smbshare/srs-9/prl_project/data")
inf_root = Path("/media/smbshare/srs-9/prl_project/training/roi_train2/run1/ensemble_output")
subject_sessions = pd.read_csv("/home/srs-9/Projects/prl_project/resources/subject-sessions.csv", index_col="sub")

suffix = "_xy20_z8"

In [17]:
with open(f"datalist{suffix}.json", 'r') as f:
    datalist = json.load(f)
    
test_data = datalist["testing"]
prl_data = []
lesion_data = []
for scan in test_data:
    scan = {k: Path(v) if isinstance(v, str) and ".nii.gz" in v else v for k, v in scan.items()}
    if  "prl" in Path(scan['label']).name:
        scan['inference'] = inf_root / Path(scan['label']).relative_to(dataroot).with_name(f"flair.phase{suffix}_ensemble.nii.gz")
        prl_data.append(scan)
    else:
        scan['inference'] = inf_root / Path(scan['label']).relative_to(dataroot).with_name(f"flair.phase{suffix}_ensemble.nii.gz")
        lesion_data.append(scan)


In [14]:
prl_data

[{'subid': 1293,
  'lesion_index': 1,
  'image': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/flair.phase_xy20_z8.nii.gz'),
  'label': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1293-20161129/1/prl_label_SRS_CH_xy20_z8.nii.gz'),
  'inference': PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train2/run21/ensemble_output/sub1293-20161129/1/flair.phase_xy20_z8_ensemble.nii.gz')},
 {'subid': 1074,
  'lesion_index': 3,
  'image': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1074-20190518/3/flair.phase_xy20_z8.nii.gz'),
  'label': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1074-20190518/3/prl_label_CH_xy20_z8.nii.gz'),
  'inference': PosixPath('/media/smbshare/srs-9/prl_project/training/roi_train2/run21/ensemble_output/sub1074-20190518/3/flair.phase_xy20_z8_ensemble.nii.gz')},
 {'subid': 1080,
  'lesion_index': 3,
  'image': PosixPath('/media/smbshare/srs-9/prl_project/data/sub1080-20180416/3/flair.phase_xy20_z8.nii.gz'),
  'label

In [18]:
prl_dice_scores = []
for scan in prl_data:
    lab_data = nib.load(scan['label']).get_fdata()
    inf_data = nib.load(scan['inference']).get_fdata()
    prl_dice_scores.append(utils.dice_score(lab_data, inf_data, seg1_val=2, seg2_val=2))

In [19]:
prl_dice_scores

[0.21082621082621084,
 0.37249283667621774,
 0.47696476964769646,
 0.6604255319148936,
 0.5487364620938628,
 0.36338028169014086,
 0.3658838071693449,
 0.5654450261780105,
 0.13846153846153847]

In [21]:
len(prl_dice_scores)

9

In [20]:
import statistics

statistics.mean(prl_dice_scores)

0.41140182940643516

In [29]:
# view_root = Path("Z:/")
view_root = Path("/media/smbshare")
for scan in prl_data:
    images = [scan['image'].with_name(im) for im in [f"flair{suffix}.nii.gz", f"phase{suffix}.nii.gz"]]
    images = [view_root/(im.relative_to("/media/smbshare")) for im in images]
    labels = [scan['label'], scan['inference']]
    labels = [view_root/(im.relative_to("/media/smbshare")) for im in labels]
    cmd = open_itksnap_workspace_cmd(images, labels=labels, rename_root=("/media/smbshare", "Z:/"))
    print(cmd)

itksnap -g Z:/srs-9/prl_project/data/sub1293-20161129/1/flair_xy20_z8.nii.gz -o Z:/srs-9/prl_project/data/sub1293-20161129/1/phase_xy20_z8.nii.gz -s Z:/srs-9/prl_project/data/sub1293-20161129/1/prl_label_SRS_CH_xy20_z8.nii.gz Z:/srs-9/prl_project/training/roi_train2/run1/ensemble_output/sub1293-20161129/1/flair.phase_xy20_z8_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1074-20190518/3/flair_xy20_z8.nii.gz -o Z:/srs-9/prl_project/data/sub1074-20190518/3/phase_xy20_z8.nii.gz -s Z:/srs-9/prl_project/data/sub1074-20190518/3/prl_label_CH_xy20_z8.nii.gz Z:/srs-9/prl_project/training/roi_train2/run1/ensemble_output/sub1074-20190518/3/flair.phase_xy20_z8_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1080-20180416/3/flair_xy20_z8.nii.gz -o Z:/srs-9/prl_project/data/sub1080-20180416/3/phase_xy20_z8.nii.gz -s Z:/srs-9/prl_project/data/sub1080-20180416/3/prl_label_SRS_xy20_z8.nii.gz Z:/srs-9/prl_project/training/roi_train2/run1/ensemble_output/sub1080-20180416/3/flair.phase_xy

In [13]:
prl_dice_scores

[0.7368421052631579,
 0.16414686825053995,
 0.06451612903225806,
 0.7192851824274014,
 0.13574660633484162]

In [14]:
lesion_dice_scores = []
for scan in lesion_data:
    lab_data = nib.load(scan['label']).get_fdata()
    inf_data = nib.load(scan['inference']).get_fdata()
    lesion_dice_scores.append(utils.dice_score(lab_data, inf_data, seg1_val=2, seg2_val=2))

In [15]:
check_lesions = []
for i, (scan, dice) in enumerate(zip(lesion_data, lesion_dice_scores)):
    if dice is not None:
        check_lesions.append(scan)

In [16]:
view_root = Path("Z:/")
for scan in check_lesions:
    images = [scan['image'].with_name(im) for im in ["flair.nii.gz", "phase.nii.gz"]]
    images = [view_root/(im.relative_to("/media/smbshare")) for im in images]
    labels = [scan['label'], scan['inference']]
    labels = [view_root/(im.relative_to("/media/smbshare")) for im in labels]
    cmd = utils.open_itksnap_workspace_cmd(images, labels=labels)
    print(cmd)

itksnap -g Z:/srs-9/prl_project/data/sub1011-20180911/3/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1011-20180911/3/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1011-20180911/3/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1011-20180911/3/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1038-20161031/2/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1038-20161031/2/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1038-20161031/2/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1038-20161031/2/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub1348-20181009/2/flair.nii.gz -o Z:/srs-9/prl_project/data/sub1348-20181009/2/phase.nii.gz -s Z:/srs-9/prl_project/data/sub1348-20181009/2/lesion.nii.gz Z:/srs-9/prl_project/training/roi_train1_segresnet/ensemble_output/sub1348-20181009/2/flair.phase_ensemble.nii.gz
itksnap -g Z:/srs-9/prl_project/data/sub2131-20190117/2/flair.nii.gz -o Z:/srs-9/p

In [16]:
subid = 2026
sesid = subject_sessions.loc[subid, "ses"]
subject_root = dataroot / f"sub{subid}-{sesid}"
l = 2
images = [subject_root/f"{l}/phase.nii.gz", subject_root/f"{l}/flair.nii.gz"]
segs = [subject_root/f"{l}/prl_label_final.nii.gz"]
cmd = utils.open_itksnap_workspace_cmd(images, labels=segs)
print(cmd)
pyperclip.copy(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/phase.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/flair.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2026-20160917/2/prl_label_final.nii.gz


In [19]:
segmentations = [
    "prl_mask_def_prob_CH.nii.gz",
    "prl_mask_def_prob_SRS.nii.gz",
    "prl_mask_def_prob_LR.nii.gz",
]
subid = 2026
sesid = subject_sessions.loc[subid, "ses"]
subject_root = dataroot / f"sub{subid}-{sesid}"

for seg in segmentations:
    if (subject_root / seg).exists():
        break
images = [subject_root/"phase.nii.gz", subject_root/"flair.nii.gz"]
segs = [subject_root/seg, subject_root/"lstai_lesion_index.nii.gz"]
cmd = utils.open_itksnap_workspace_cmd(images, labels=segs)
print(cmd)
pyperclip.copy(cmd)

itksnap -g /media/smbshare/srs-9/prl_project/data/sub2026-20160917/phase.nii.gz -o /media/smbshare/srs-9/prl_project/data/sub2026-20160917/flair.nii.gz -s /media/smbshare/srs-9/prl_project/data/sub2026-20160917/prl_mask_def_prob_SRS.nii.gz /media/smbshare/srs-9/prl_project/data/sub2026-20160917/lstai_lesion_index.nii.gz
